# 🧠 01 — Neural Network Architecture for Fraud Detection

**Fraud Detection Capstone — Sprint 2: Deep Learning & Advanced Modelling**

## 🎯 Goal
Design an improved dense-network architecture for the fraud data — deeper/wider than the Sprint 1 baseline network, with regularization built in from the start (Batch Normalization + Dropout), and compare its *untrained, structural* capacity reasoning before training in the next notebook.

**Data:** uses the processed splits saved by Sprint 1's `02_data_preprocessing_feature_engineering.ipynb` (`data/processed/{train,val,test}.csv`).


In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

proc_dir = Path("data/processed")
train = pd.read_csv(proc_dir / "train.csv")
val = pd.read_csv(proc_dir / "val.csv")
test = pd.read_csv(proc_dir / "test.csv")

X_train, y_train = train.drop(columns=["Class"]), train["Class"]
X_val, y_val = val.drop(columns=["Class"]), val["Class"]
X_test, y_test = test.drop(columns=["Class"]), test["Class"]

n_features = X_train.shape[1]
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Features: {n_features}")


I0000 00:00:1788704304.576262    1379 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788704304.626183    1379 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1788704306.214008    1379 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Train: (3506, 32) | Val: (751, 32) | Test: (752, 32)
Features: 32


## 1. Sprint 1 Baseline Architecture (Reference)

```
Dense(32, relu) -> Dense(16, relu) -> Dense(1, sigmoid)
```

No regularization, no normalization. Val results: accuracy 0.9993, precision 0.857, recall 0.75, F1 0.80.


## 2. Sprint 2 Architecture

Changes from the Sprint 1 baseline, each with a reason:

| Change | Why |
|---|---|
| Deeper (4 hidden layers vs. 2) | More capacity to learn the fraud/normal boundary, which is subtle in PCA-anonymized features |
| **Batch Normalization** after each hidden layer | Stabilizes training as the network gets deeper |
| **Dropout** (0.3 → 0.2 → 0.1, decreasing toward the output) | Reduces overfitting; lighter near the output since that layer needs less regularization |
| Wider first layer (64 vs. 32 units) | More room to capture patterns before compressing down |

This is a **capacity change**, not yet a training-process change (loss weighting, resampling) — those are separate, deliberate topics for `02_nn_training_regularization.ipynb` and Sprint 3.


In [2]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout

def build_sprint2_model(n_features):
    return Sequential([
        Dense(64, activation="relu", input_shape=(n_features,)),
        BatchNormalization(),
        Dropout(0.3),

        Dense(32, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),

        Dense(16, activation="relu"),
        BatchNormalization(),
        Dropout(0.1),

        Dense(1, activation="sigmoid"),
    ])

model = build_sprint2_model(n_features)
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,185 (20.25 KB)

 Trainable params: 4,961 (19.38 KB)

 Non-trainable params: 224 (896.00 B)

## 3. Parameter Count vs. Sprint 1 Baseline

In [3]:
baseline_model = Sequential([
    Dense(32, activation="relu", input_shape=(n_features,)),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid"),
])

sprint1_params = baseline_model.count_params()
sprint2_params = model.count_params()

print(f"Sprint 1 baseline params: {sprint1_params:,}")
print(f"Sprint 2 architecture params: {sprint2_params:,}")
print(f"Increase: {sprint2_params - sprint1_params:,} ({(sprint2_params/sprint1_params - 1)*100:.0f}% more)")


Sprint 1 baseline params: 1,601
Sprint 2 architecture params: 5,185
Increase: 3,584 (224% more)


In [4]:
model.save("models/sprint2_architecture_untrained.keras")
print("Architecture saved (untrained) -> models/sprint2_architecture_untrained.keras")


Architecture saved (untrained) -> models/sprint2_architecture_untrained.keras


## 📝 Summary

Defined and saved (untrained) an improved architecture: deeper, wider first layer, with Batch Normalization and decreasing Dropout for regularization. More capacity than Sprint 1's baseline, at the cost of more parameters to train carefully — which is exactly the training-process question the next notebook addresses.

**Next:** `02_nn_training_regularization.ipynb` — train this architecture with EarlyStopping/ModelCheckpoint and examine what the added regularization actually buys.
